In [7]:
"""
Decoding Analysis: spatial_label & temporal_label from iEEG IRASA features

Data unit  : one recalled WORD per prediction
             (rows are word x region x freq -> pivot region x freq -> wide vector)
CV method  : Leave-One-TRIAL-Out per subject
             - hold out all words from one trial as test set
             - train on all words from remaining trials
             - this respects trial-level non-independence of words
Feature sets:
  - encoding_only  : encoding_frac_ssl, encoding_osc_ssl
  - retrieval_only : retrieval_frac_ssl, retrieval_osc_ssl
  - combined       : all four columns
Classifier : Logistic Regression (balanced class weights)

Fixes applied
-------------
1. Permutation bug: trial-level shuffle now remaps trial IDs while keeping
   per-word labels intact, preserving within-trial label heterogeneity.
2. Imputation moved inside CV loop (fit on train fold only) to prevent
   test-set leakage via median imputation.
3. Outer parallelism kept over subjects; inner permutation loop runs serially
   per subject (n_jobs=1) to avoid nested-process overhead.
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from scipy.stats import ttest_1samp
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH   = "ALL_SUBJECTS_irasa_labeled.csv"
LABELS      = ["spatial_label", "temporal_label"]
N_PERM      = 1000
RANDOM_SEED = 42
N_JOBS      = -1   # parallelise over subjects; permutations run serially inside

FEATURE_SETS = {
    "encoding_only":  ["encoding_frac_ssl", "encoding_osc_ssl"],
    "retrieval_only": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
    "combined":       ["encoding_frac_ssl",  "encoding_osc_ssl",
                       "retrieval_frac_ssl", "retrieval_osc_ssl"],
}

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df["region"] = df["region"].astype(str)

print("Loaded {} rows | {} subjects".format(len(df), df["subject"].nunique()))
print("Regions : {}".format(sorted(df["region"].unique())))
print("Freqs   : {}".format(sorted(df["freq_hz"].unique())))
print()


# ── Feature builder ───────────────────────────────────────────────────────────
def build_word_feature_matrix(subj_df, power_cols):
    """
    Pivot word x region x freq rows into one wide vector per word.

    Returns
    -------
    X    : np.ndarray  shape (n_words, n_features)  — NaNs retained here;
                        imputation happens inside the CV loop.
    meta : pd.DataFrame  with columns [trial, recalled_word,
                                        spatial_label, temporal_label]
    """
    # NOTE: subj_df is long-format (one row per word × region × freq), so
    # duplicates on (trial, recalled_word) are expected and correct.
    # Uniqueness is enforced at the word level after groupby below.

    meta = (
        subj_df
        .groupby(["trial", "recalled_word"])[["spatial_label", "temporal_label"]]
        .first()
        .reset_index()
    )
    meta = meta.reset_index(drop=True)
    meta["word_idx"] = meta.index

    subj_df = subj_df.merge(
        meta[["trial", "recalled_word", "word_idx"]],
        on=["trial", "recalled_word"],
        how="left"
    )

    pivot_frames = []
    for pcol in power_cols:
        piv = subj_df.pivot_table(
            index="word_idx",
            columns=["region", "freq_hz"],
            values=pcol,
            aggfunc="mean"
        )
        piv.columns = ["{}_{}_{}Hz".format(pcol, r, f) for r, f in piv.columns]
        pivot_frames.append(piv)

    X_df = pd.concat(pivot_frames, axis=1)
    X_df = X_df.reindex(meta["word_idx"])
    X_df = X_df.dropna(axis=1, how="all")
    # NOTE: do NOT fillna here — imputation happens inside the CV loop

    return X_df.values.astype(float), meta.reset_index(drop=True)


# ── Leave-one-trial-out decoder ───────────────────────────────────────────────
def _make_pipeline():
    """Scaler + median imputer + LogReg, all fit on train fold only."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("clf",     LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            solver="lbfgs",
            random_state=RANDOM_SEED,
        )),
    ])


def loto_decode(X, y, trials):
    """
    Leave-one-trial-out cross-validation.
    Train on all words outside held-out trial; predict words inside it.

    FIX 2: imputation (median) is fit on the training fold only via Pipeline,
    preventing any leakage from test words into feature scaling/imputation.

    Returns None if any fold has fewer than 2 classes in training set.

    Parameters
    ----------
    X      : np.ndarray (n_words, n_features)  — may contain NaNs
    y      : np.ndarray (n_words,) binary labels
    trials : np.ndarray (n_words,) trial index per word
    """
    if len(np.unique(y)) < 2:
        return None

    unique_trials = np.unique(trials)
    if len(unique_trials) < 2:
        return None

    probs = np.full(len(y), np.nan)
    preds = np.full(len(y), np.nan)

    for held_trial in unique_trials:
        test_mask  = trials == held_trial
        train_mask = ~test_mask

        y_train = y[train_mask]
        if len(np.unique(y_train)) < 2:
            continue

        pipe = _make_pipeline()
        pipe.fit(X[train_mask], y_train)
        probs[test_mask] = pipe.predict_proba(X[test_mask])[:, 1]
        preds[test_mask] = pipe.predict(X[test_mask])

    valid = ~np.isnan(probs)
    if valid.sum() < 2 or len(np.unique(y[valid])) < 2:
        return None

    return {
        "auc":   roc_auc_score(y[valid], probs[valid]),
        "bacc":  balanced_accuracy_score(y[valid], preds[valid]),
        "probs": probs,
        "preds": preds,
    }


# ── Permutation test ──────────────────────────────────────────────────────────
def _one_perm(X, y, trials, unique_tr, seed):
    """
    Run a single permutation; called in a serial loop inside permutation_auc.

    FIX 1: shuffle trial IDs (remap each trial to a different trial's position)
    while keeping per-word labels intact.  This preserves within-trial label
    heterogeneity — words in a trial keep their own labels; we only change
    which fold they are assigned to during CV.

    The previous implementation assigned every word in a trial the label of
    the *first* word in whatever trial it was remapped to, collapsing
    within-trial label variation and producing an overly narrow null.
    """
    rng = np.random.default_rng(seed)
    # Permute the trial IDs themselves; word→label mapping is unchanged
    shuffled_tr = rng.permutation(unique_tr)
    trial_remap = dict(zip(unique_tr, shuffled_tr))
    perm_trials = np.array([trial_remap[t] for t in trials])
    # y is NOT permuted — only the trial fold memberships change
    res = loto_decode(X, y, perm_trials)
    return res["auc"] if res is not None else np.nan


def permutation_auc(X, y, trials, n_perm=N_PERM):
    """
    Trial-level permutation test.
    Returns p-value = proportion of null AUCs >= observed AUC.
    Runs serially (n_jobs=1) because the outer loop is already parallelised
    over subjects.
    """
    obs = loto_decode(X, y, trials)
    if obs is None:
        return np.nan

    unique_tr = np.unique(trials)
    seeds     = np.arange(RANDOM_SEED, RANDOM_SEED + n_perm)

    null_aucs = [
        _one_perm(X, y, trials, unique_tr, int(s))
        for s in seeds
    ]
    null_aucs = np.array([v for v in null_aucs if not np.isnan(v)])

    if len(null_aucs) == 0:
        return np.nan
    return float(np.mean(null_aucs >= obs["auc"]))


# ── Per-subject worker ────────────────────────────────────────────────────────
def run_subject(subj, subj_df):
    """
    Run all feature sets x labels for one subject.
    Returns a list of result dicts.
    """
    rows = []
    print("Starting subject: {}".format(subj))

    for feat_name, power_cols in FEATURE_SETS.items():
        X, meta = build_word_feature_matrix(subj_df, power_cols)
        trials  = meta["trial"].values
        n_words = len(meta)

        for label in LABELS:
            y     = meta[label].values.astype(int)
            n_pos = int(y.sum())

            res = loto_decode(X, y, trials)
            if res is None:
                rows.append({
                    "subject":     subj,
                    "feature_set": feat_name,
                    "label":       label,
                    "n_words":     n_words,
                    "n_pos":       n_pos,
                    "auc":         np.nan,
                    "bacc":        np.nan,
                    "perm_p":      np.nan,
                })
                continue

            p_val = permutation_auc(X, y, trials, n_perm=N_PERM)

            print("  {} | {} | {} -> AUC={:.3f}  BalAcc={:.3f}  perm_p={:.3f}".format(
                subj, feat_name, label, res["auc"], res["bacc"], p_val))

            rows.append({
                "subject":     subj,
                "feature_set": feat_name,
                "label":       label,
                "n_words":     n_words,
                "n_pos":       n_pos,
                "auc":         res["auc"],
                "bacc":        res["bacc"],
                "perm_p":      p_val,
            })

    return rows


# ── Main loop (parallel over subjects) ───────────────────────────────────────
subjects = sorted(df["subject"].unique())
print("Running {} subjects in parallel (n_jobs={})...\n".format(
    len(subjects), N_JOBS))

all_rows = Parallel(n_jobs=N_JOBS)(
    delayed(run_subject)(subj, df[df["subject"] == subj].copy())
    for subj in subjects
)

# flatten list of lists
results = [row for subj_rows in all_rows for row in subj_rows]

# ── Save & group summary ──────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df.to_csv("decoding_results_per_subject.csv", index=False)

print()
print("=" * 60)
print("GROUP-LEVEL SUMMARY  (one-sample t vs AUC = 0.5)")
print("=" * 60)

for label in LABELS:
    print("\n  Label: {}".format(label))
    print("  {:<20} {:>4} {:>10} {:>7} {:>7} {:>8} {:>7}".format(
        "Feature set", "n", "Mean AUC", "SD", "t", "p", "n_sig"))
    print("  " + "-" * 62)
    for feat_name in FEATURE_SETS:
        sub = results_df[
            (results_df["label"] == label) &
            (results_df["feature_set"] == feat_name)
        ].dropna(subset=["auc"])
        if len(sub) < 2:
            continue
        aucs    = sub["auc"].values
        t, p    = ttest_1samp(aucs, popmean=0.5)
        n_sig   = int((sub["perm_p"] < 0.05).sum())
        print("  {:<20} {:>4} {:>10.3f} {:>7.3f} {:>7.3f} {:>8.4f} {:>4}/{}".format(
            feat_name, len(aucs), aucs.mean(), aucs.std(),
            t, p, n_sig, len(aucs)))

print()
print("Results saved -> decoding_results_per_subject.csv")

Loaded 27162 rows | 26 subjects
Regions : ['LHP', 'RHP']
Freqs   : [3.0, 3.74, 4.67, 5.82, 7.26, 9.05, 11.28, 14.07, 17.55, 21.88, 27.29, 34.03, 42.44, 52.92, 66.0, 82.31, 102.64, 128.0]

Running 26 subjects in parallel (n_jobs=-1)...



KeyboardInterrupt: 

In [8]:
"""
Decoding Analysis: spatial_label & temporal_label from iEEG IRASA features
(Permutation testing removed for speed)

Data unit  : one recalled WORD per prediction
CV method  : Leave-One-TRIAL-Out per subject
Feature sets:
  - encoding_only  : encoding_frac_ssl, encoding_osc_ssl
  - retrieval_only : retrieval_frac_ssl, retrieval_osc_ssl
  - combined       : all four columns
Classifier : Logistic Regression (balanced class weights)
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from scipy.stats import ttest_1samp
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH   = "ALL_SUBJECTS_irasa_labeled.csv"
LABELS      = ["spatial_label", "temporal_label"]
RANDOM_SEED = 42
N_JOBS      = -1

FEATURE_SETS = {
    "encoding_only":  ["encoding_frac_ssl", "encoding_osc_ssl"],
    "retrieval_only": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
    "combined":       ["encoding_frac_ssl",  "encoding_osc_ssl",
                       "retrieval_frac_ssl", "retrieval_osc_ssl"],
}

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df["region"] = df["region"].astype(str)

print("Loaded {} rows | {} subjects".format(len(df), df["subject"].nunique()))
print("Regions : {}".format(sorted(df["region"].unique())))
print("Freqs   : {}".format(sorted(df["freq_hz"].unique())))
print()


# ── Feature builder ───────────────────────────────────────────────────────────
def build_word_feature_matrix(subj_df, power_cols):
    meta = (
        subj_df
        .groupby(["trial", "recalled_word"])[["spatial_label", "temporal_label"]]
        .first()
        .reset_index()
    )
    meta = meta.reset_index(drop=True)
    meta["word_idx"] = meta.index

    subj_df = subj_df.merge(
        meta[["trial", "recalled_word", "word_idx"]],
        on=["trial", "recalled_word"],
        how="left"
    )

    pivot_frames = []
    for pcol in power_cols:
        piv = subj_df.pivot_table(
            index="word_idx",
            columns=["region", "freq_hz"],
            values=pcol,
            aggfunc="mean"
        )
        piv.columns = ["{}_{}_{}Hz".format(pcol, r, f) for r, f in piv.columns]
        pivot_frames.append(piv)

    X_df = pd.concat(pivot_frames, axis=1)
    X_df = X_df.reindex(meta["word_idx"])
    X_df = X_df.dropna(axis=1, how="all")

    return X_df.values.astype(float), meta.reset_index(drop=True)


# ── Leave-one-trial-out decoder ───────────────────────────────────────────────
def _make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("clf",     LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            solver="lbfgs",
            random_state=RANDOM_SEED,
        )),
    ])


def loto_decode(X, y, trials):
    if len(np.unique(y)) < 2:
        return None

    unique_trials = np.unique(trials)
    if len(unique_trials) < 2:
        return None

    probs = np.full(len(y), np.nan)
    preds = np.full(len(y), np.nan)

    for held_trial in unique_trials:
        test_mask  = trials == held_trial
        train_mask = ~test_mask

        y_train = y[train_mask]
        if len(np.unique(y_train)) < 2:
            continue

        pipe = _make_pipeline()
        pipe.fit(X[train_mask], y_train)
        probs[test_mask] = pipe.predict_proba(X[test_mask])[:, 1]
        preds[test_mask] = pipe.predict(X[test_mask])

    valid = ~np.isnan(probs)
    if valid.sum() < 2 or len(np.unique(y[valid])) < 2:
        return None

    return {
        "auc":   roc_auc_score(y[valid], probs[valid]),
        "bacc":  balanced_accuracy_score(y[valid], preds[valid]),
        "probs": probs,
        "preds": preds,
    }


# ── Per-subject worker ────────────────────────────────────────────────────────
def run_subject(subj, subj_df):
    rows = []
    print("  Starting: {}".format(subj), flush=True)

    for feat_name, power_cols in FEATURE_SETS.items():
        X, meta = build_word_feature_matrix(subj_df, power_cols)
        trials  = meta["trial"].values
        n_words = len(meta)

        for label in LABELS:
            y     = meta[label].values.astype(int)
            n_pos = int(y.sum())

            res = loto_decode(X, y, trials)
            if res is None:
                rows.append({
                    "subject":     subj,
                    "feature_set": feat_name,
                    "label":       label,
                    "n_words":     n_words,
                    "n_pos":       n_pos,
                    "auc":         np.nan,
                    "bacc":        np.nan,
                })
                continue

            print("  {} | {} | {} -> AUC={:.3f}  BalAcc={:.3f}".format(
                subj, feat_name, label, res["auc"], res["bacc"]), flush=True)

            rows.append({
                "subject":     subj,
                "feature_set": feat_name,
                "label":       label,
                "n_words":     n_words,
                "n_pos":       n_pos,
                "auc":         res["auc"],
                "bacc":        res["bacc"],
            })

    return rows


# ── Main loop (parallel over subjects) ───────────────────────────────────────
subjects = sorted(df["subject"].unique())
print("Running {} subjects in parallel (n_jobs={})...\n".format(
    len(subjects), N_JOBS))

all_rows = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(run_subject)(subj, df[df["subject"] == subj].copy())
    for subj in subjects
)

results = [row for subj_rows in all_rows for row in subj_rows]

# ── Save & group summary ──────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df.to_csv("decoding_results_per_subject.csv", index=False)

print()
print("=" * 60)
print("GROUP-LEVEL SUMMARY  (one-sample t vs AUC = 0.5)")
print("=" * 60)

for label in LABELS:
    print("\n  Label: {}".format(label))
    print("  {:<20} {:>4} {:>10} {:>7} {:>7} {:>8}".format(
        "Feature set", "n", "Mean AUC", "SD", "t", "p"))
    print("  " + "-" * 54)
    for feat_name in FEATURE_SETS:
        sub = results_df[
            (results_df["label"] == label) &
            (results_df["feature_set"] == feat_name)
        ].dropna(subset=["auc"])
        if len(sub) < 2:
            continue
        aucs = sub["auc"].values
        t, p = ttest_1samp(aucs, popmean=0.5)
        print("  {:<20} {:>4} {:>10.3f} {:>7.3f} {:>7.3f} {:>8.4f}".format(
            feat_name, len(aucs), aucs.mean(), aucs.std(), t, p))

print()
print("Results saved -> decoding_results_per_subject.csv")

Loaded 27162 rows | 26 subjects
Regions : ['LHP', 'RHP']
Freqs   : [3.0, 3.74, 4.67, 5.82, 7.26, 9.05, 11.28, 14.07, 17.55, 21.88, 27.29, 34.03, 42.44, 52.92, 66.0, 82.31, 102.64, 128.0]

Running 26 subjects in parallel (n_jobs=-1)...



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:    2.4s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    5.1s



GROUP-LEVEL SUMMARY  (one-sample t vs AUC = 0.5)

  Label: spatial_label
  Feature set             n   Mean AUC      SD       t        p
  ------------------------------------------------------
  encoding_only           4      0.328   0.248  -1.201   0.3158
  retrieval_only          4      0.294   0.156  -2.288   0.1061
  combined                4      0.305   0.229  -1.469   0.2381

  Label: temporal_label
  Feature set             n   Mean AUC      SD       t        p
  ------------------------------------------------------
  encoding_only           8      0.397   0.182  -1.496   0.1784
  retrieval_only          8      0.338   0.179  -2.404   0.0472
  combined                8      0.358   0.168  -2.234   0.0606

Results saved -> decoding_results_per_subject.csv


[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    5.8s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    5.8s finished


In [14]:
"""
Between-subject decoding: predict spatial/temporal clustering score
from subject-averaged iEEG IRASA features.

CV method  : Leave-One-Subject-Out (LOSO)
Model      : RidgeCV (alpha selected by inner LOO-CV on training fold)
Features   : subject-averaged encoding/retrieval frac_ssl and osc_ssl
Target     : continuous spatial/temporal clustering score (subject mean)
Statistics : Pearson r, p-value, R²
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH = "ALL_SUBJECTS_irasa_labeled.csv"
TARGETS = ["SC_t", "TC_t"]
RANDOM_SEED = 42

FEATURE_SETS = {
    "encoding_only":  ["encoding_frac_ssl", "encoding_osc_ssl"],
    "retrieval_only": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
    "combined":       ["encoding_frac_ssl",  "encoding_osc_ssl",
                       "retrieval_frac_ssl", "retrieval_osc_ssl"],
}

ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df["region"] = df["region"].astype(str)
subjects = sorted(df["subject"].unique())
print("Loaded {} rows | {} subjects".format(len(df), len(subjects)))

# ── Build subject-level feature matrix ───────────────────────────────────────
def build_subject_matrix(df, power_cols):
    """
    Average all word x region x freq rows per subject into one wide vector.
    Returns:
        X_df : pd.DataFrame, shape (n_subjects, n_features), index=subject
        y_df : pd.DataFrame, shape (n_subjects, n_targets), index=subject
    """
    pivot_frames = []
    for pcol in power_cols:
        piv = df.pivot_table(
            index="subject",
            columns=["region", "freq_hz"],
            values=pcol,
            aggfunc="mean"
        )
        piv.columns = ["{}_{}_{}Hz".format(pcol, r, f) for r, f in piv.columns]
        pivot_frames.append(piv)

    X_df = pd.concat(pivot_frames, axis=1)
    X_df = X_df.dropna(axis=1, how="all")

    # subject-level target: mean clustering score across recalled words
    y_df = df.groupby("subject")[TARGETS].mean()

    # align
    X_df, y_df = X_df.align(y_df, join="inner", axis=0)
    return X_df, y_df


# ── LOSO decoder ─────────────────────────────────────────────────────────────
def loso_ridge(X, y, subjects):
    """
    Leave-one-subject-out Ridge regression.
    Returns predicted y for each left-out subject.
    """
    X = np.array(X)
    y = np.array(y)
    subjects = np.array(subjects)
    preds = np.full(len(y), np.nan)

    for i, subj in enumerate(subjects):
        test_mask  = subjects == subj
        train_mask = ~test_mask

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("ridge",   RidgeCV(alphas=ALPHAS, cv=None)),  # cv=None -> LOO on train
        ])
        pipe.fit(X[train_mask], y[train_mask])
        preds[i] = pipe.predict(X[test_mask])[0]

    return preds


# ── Run all feature sets x targets ───────────────────────────────────────────
results = []

for feat_name, power_cols in FEATURE_SETS.items():
    print("\nFeature set: {}".format(feat_name))

    X_df, y_df = build_subject_matrix(df, power_cols)
    X = X_df.values
    subj_arr = np.array(X_df.index)

    for target in TARGETS:
        y = y_df[target].values

        # drop subjects with NaN target
        valid = ~np.isnan(y)
        X_v, y_v, s_v = X[valid], y[valid], subj_arr[valid]

        if len(y_v) < 4:
            print("  {} -> too few subjects ({})".format(target, len(y_v)))
            continue

        preds = loso_ridge(X_v, y_v, s_v)

        valid_pred = ~np.isnan(preds)
        r, p = pearsonr(y_v[valid_pred], preds[valid_pred])
        r2 = r ** 2

        print("  {:35s} | n={:2d} | r={:+.3f} | R²={:.3f} | p={:.4f}".format(
            target, int(valid_pred.sum()), r, r2, p))

        results.append({
            "feature_set": feat_name,
            "target":      target,
            "n_subjects":  int(valid_pred.sum()),
            "r":           r,
            "r2":          r2,
            "p":           p,
        })

# ── Save ──────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df.to_csv("decoding_between_subject_ridge.csv", index=False)

print("\n\nFull results:")
print(results_df.to_string(index=False))
print("\nSaved -> decoding_between_subject_ridge.csv")

Loaded 27162 rows | 26 subjects

Feature set: encoding_only
  SC_t                                | n=25 | r=-0.110 | R²=0.012 | p=0.6017
  TC_t                                | n=21 | r=+0.330 | R²=0.109 | p=0.1445

Feature set: retrieval_only
  SC_t                                | n=25 | r=-0.769 | R²=0.591 | p=0.0000
  TC_t                                | n=21 | r=+0.532 | R²=0.283 | p=0.0131

Feature set: combined
  SC_t                                | n=25 | r=-0.606 | R²=0.367 | p=0.0013
  TC_t                                | n=21 | r=+0.492 | R²=0.242 | p=0.0236


Full results:
   feature_set target  n_subjects         r       r2        p
 encoding_only   SC_t          25 -0.109693 0.012032 0.601696
 encoding_only   TC_t          21  0.329606 0.108640 0.144540
retrieval_only   SC_t          25 -0.768618 0.590774 0.000007
retrieval_only   TC_t          21  0.531742 0.282750 0.013105
      combined   SC_t          25 -0.605583 0.366731 0.001337
      combined   TC_t          2

In [19]:
"""
Within-subject decoding: predict SC_t / TC_t from word-level iEEG IRASA features.

CV method  : Leave-One-Trial-Out per subject
Model      : RidgeCV (alpha selected by inner LOO-CV on training fold)
Features   : word-level encoding/retrieval frac_ssl and osc_ssl
             pivoted across regions x frequencies
Target     : continuous SC_t / TC_t per recalled word
Statistics : Pearson r, R², p-value per subject -> group t-test vs r=0
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from scipy.stats import pearsonr, ttest_1samp
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH   = "ALL_SUBJECTS_irasa_labeled.csv"
TARGETS     = ["SC_t", "TC_t"]
RANDOM_SEED = 42
N_JOBS      = -1

FEATURE_SETS = {
    "encoding_only":  ["encoding_frac_ssl", "encoding_osc_ssl"],
    "retrieval_only": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
    "combined":       ["encoding_frac_ssl",  "encoding_osc_ssl",
                       "retrieval_frac_ssl", "retrieval_osc_ssl"],
}

ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df["region"] = df["region"].astype(str)
subjects = sorted(df["subject"].unique())
print("Loaded {} rows | {} subjects".format(len(df), len(subjects)))
print("Regions : {}".format(sorted(df["region"].unique())))
print("Freqs   : {}".format(sorted(df["freq_hz"].unique())))
print()


# ── Feature builder ───────────────────────────────────────────────────────────
def build_word_feature_matrix(subj_df, power_cols):
    """
    Pivot word x region x freq rows into one wide vector per word.
    Returns X (n_words, n_features), meta (trial, SC_t, TC_t per word).
    """
    meta = (
        subj_df
        .groupby(["trial", "recalled_word"])[["SC_t", "TC_t"]]
        .first()
        .reset_index()
    )
    meta = meta.reset_index(drop=True)
    meta["word_idx"] = meta.index

    subj_df = subj_df.merge(
        meta[["trial", "recalled_word", "word_idx"]],
        on=["trial", "recalled_word"],
        how="left"
    )

    pivot_frames = []
    for pcol in power_cols:
        piv = subj_df.pivot_table(
            index="word_idx",
            columns=["region", "freq_hz"],
            values=pcol,
            aggfunc="mean"
        )
        piv.columns = ["{}_{}_{}Hz".format(pcol, r, f) for r, f in piv.columns]
        pivot_frames.append(piv)

    X_df = pd.concat(pivot_frames, axis=1)
    X_df = X_df.reindex(meta["word_idx"])
    X_df = X_df.dropna(axis=1, how="all")

    return X_df.values.astype(float), meta.reset_index(drop=True)


# ── Leave-one-trial-out Ridge ─────────────────────────────────────────────────
def loto_ridge(X, y, trials):
    """
    Leave-one-trial-out Ridge regression.
    Returns predicted y for each word, or None if underpowered.
    """
    unique_trials = np.unique(trials)
    if len(unique_trials) < 3:
        return None
    if np.nanstd(y) == 0:
        return None

    preds = np.full(len(y), np.nan)

    for held_trial in unique_trials:
        test_mask  = trials == held_trial
        train_mask = ~test_mask

        X_train, y_train = X[train_mask], y[train_mask]
        X_test           = X[test_mask]

        # drop rows where y_train is NaN or inf
        finite_mask = np.isfinite(y_train)
        if finite_mask.sum() < 3:
            continue
        X_train = X_train[finite_mask]
        y_train = y_train[finite_mask]

        if np.std(y_train) == 0:
            continue

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("ridge",   RidgeCV(alphas=ALPHAS, cv=None)),
        ])
        pipe.fit(X_train, y_train)
        preds[test_mask] = pipe.predict(X_test)

    return preds


# ── Per-subject worker ────────────────────────────────────────────────────────
def run_subject(subj, subj_df):
    rows = []
    print("  Starting: {}".format(subj), flush=True)

    for feat_name, power_cols in FEATURE_SETS.items():
        X, meta = build_word_feature_matrix(subj_df, power_cols)
        trials  = meta["trial"].values

        for target in TARGETS:
            y = meta[target].values.astype(float)

            # drop words with NaN target
            valid = ~np.isnan(y)
            if valid.sum() < 6:
                rows.append({"subject": subj, "feature_set": feat_name,
                             "target": target, "n_words": int(valid.sum()),
                             "r": np.nan, "r2": np.nan, "p": np.nan})
                continue

            preds = loto_ridge(X[valid], y[valid], trials[valid])
            if preds is None:
                rows.append({"subject": subj, "feature_set": feat_name,
                             "target": target, "n_words": int(valid.sum()),
                             "r": np.nan, "r2": np.nan, "p": np.nan})
                continue

            valid_pred = np.isfinite(preds) & np.isfinite(y[valid])
            if valid_pred.sum() < 4 or np.std(preds[valid_pred]) == 0 or np.std(y[valid][valid_pred]) == 0:
                rows.append({"subject": subj, "feature_set": feat_name,
                             "target": target, "n_words": int(valid.sum()),
                             "r": np.nan, "r2": np.nan, "p": np.nan})
                continue

            r, p = pearsonr(y[valid][valid_pred], preds[valid_pred])
            r2   = r ** 2

            print("  {} | {} | {} -> r={:+.3f}  R²={:.3f}  p={:.4f}".format(
                subj, feat_name, target, r, r2, p), flush=True)

            rows.append({
                "subject":     subj,
                "feature_set": feat_name,
                "target":      target,
                "n_words":     int(valid.sum()),
                "r":           r,
                "r2":          r2,
                "p":           p,
            })

    return rows


# ── Main loop (parallel over subjects) ───────────────────────────────────────
all_rows = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(run_subject)(subj, df[df["subject"] == subj].copy())
    for subj in subjects
)

results = [row for subj_rows in all_rows for row in subj_rows]
results_df = pd.DataFrame(results)
results_df.to_csv("decoding_within_subject_ridge.csv", index=False)

# ── Group summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("GROUP-LEVEL SUMMARY  (one-sample t vs r = 0)")
print("=" * 65)

for target in TARGETS:
    print("\n  Target: {}".format(target))
    print("  {:<20} {:>4} {:>8} {:>7} {:>7} {:>8} {:>7}".format(
        "Feature set", "n", "Mean r", "SD", "t", "p", "n_sig"))
    print("  " + "-" * 58)
    for feat_name in FEATURE_SETS:
        sub = results_df[
            (results_df["target"]      == target) &
            (results_df["feature_set"] == feat_name)
        ].dropna(subset=["r"])
        if len(sub) < 2:
            continue
        rs    = sub["r"].values
        t, p  = ttest_1samp(rs, popmean=0)
        n_sig = int((sub["p"] < 0.05).sum())
        print("  {:<20} {:>4} {:>8.3f} {:>7.3f} {:>7.3f} {:>8.4f} {:>4}/{}".format(
            feat_name, len(rs), rs.mean(), rs.std(), t, p,
            n_sig, len(rs)))

print()
print("Saved -> decoding_within_subject_ridge.csv")

Loaded 27162 rows | 26 subjects
Regions : ['LHP', 'RHP']
Freqs   : [3.0, 3.74, 4.67, 5.82, 7.26, 9.05, 11.28, 14.07, 17.55, 21.88, 27.29, 34.03, 42.44, 52.92, 66.0, 82.31, 102.64, 128.0]



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    4.3s



GROUP-LEVEL SUMMARY  (one-sample t vs r = 0)

  Target: SC_t
  Feature set             n   Mean r      SD       t        p   n_sig
  ----------------------------------------------------------
  encoding_only          23   -0.449   0.317  -6.636   0.0000   14/23
  retrieval_only         23   -0.412   0.226  -8.554   0.0000   14/23
  combined               23   -0.458   0.281  -7.648   0.0000   17/23

  Target: TC_t
  Feature set             n   Mean r      SD       t        p   n_sig
  ----------------------------------------------------------
  encoding_only          23   -0.355   0.361  -4.613   0.0001   13/23
  retrieval_only         23   -0.452   0.300  -7.067   0.0000   15/23
  combined               23   -0.324   0.373  -4.072   0.0005   15/23

Saved -> decoding_within_subject_ridge.csv


[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    5.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    5.0s finished


In [20]:
"""
Within-subject decoding + cross-decoding: SC_t ↔ TC_t from word-level iEEG IRASA features.

CV method  : Leave-One-Trial-Out per subject
Model      : RidgeCV (alpha selected by inner LOO-CV on training fold)
Features   : word-level encoding/retrieval frac_ssl and osc_ssl
             pivoted across regions x frequencies
Targets    : SC_t, TC_t (continuous, per recalled word)

Decoding modes
──────────────
  within  : train on SC_t → test on SC_t  (and TC_t → TC_t)
  cross   : train on SC_t → test on TC_t  (and TC_t → SC_t)

Statistics : Pearson r, R², p-value per subject → group one-sample t vs r = 0
             Cross vs within comparison: paired t-test per feature set
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from scipy.stats import pearsonr, ttest_1samp, ttest_rel
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH   = "ALL_SUBJECTS_irasa_labeled.csv"
RANDOM_SEED = 42
N_JOBS      = -1

FEATURE_SETS = {
    "encoding_only":  ["encoding_frac_ssl",  "encoding_osc_ssl"],
    "retrieval_only": ["retrieval_frac_ssl",  "retrieval_osc_ssl"],
    "combined":       ["encoding_frac_ssl",   "encoding_osc_ssl",
                       "retrieval_frac_ssl",  "retrieval_osc_ssl"],
}

# All four train→test combinations
DECODE_PAIRS = [
    ("SC_t", "SC_t"),   # within SC
    ("TC_t", "TC_t"),   # within TC
    ("SC_t", "TC_t"),   # cross: SC trains, TC tested
    ("TC_t", "SC_t"),   # cross: TC trains, SC tested
]

ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df["region"] = df["region"].astype(str)
subjects = sorted(df["subject"].unique())
print("Loaded {} rows | {} subjects".format(len(df), len(subjects)))
print("Regions : {}".format(sorted(df["region"].unique())))
print("Freqs   : {}".format(sorted(df["freq_hz"].unique())))
print()


# ── Feature builder ───────────────────────────────────────────────────────────
def build_word_feature_matrix(subj_df, power_cols):
    """
    Pivot word × region × freq rows into one wide vector per word.
    Returns:
        X    : (n_words, n_features) float array
        meta : DataFrame with columns [trial, recalled_word, word_idx, SC_t, TC_t]
    """
    meta = (
        subj_df
        .groupby(["trial", "recalled_word"])[["SC_t", "TC_t"]]
        .first()
        .reset_index()
        .reset_index(drop=True)
    )
    meta["word_idx"] = meta.index

    subj_df = subj_df.merge(
        meta[["trial", "recalled_word", "word_idx"]],
        on=["trial", "recalled_word"],
        how="left"
    )

    pivot_frames = []
    for pcol in power_cols:
        piv = subj_df.pivot_table(
            index="word_idx",
            columns=["region", "freq_hz"],
            values=pcol,
            aggfunc="mean"
        )
        piv.columns = ["{}_{}_{}Hz".format(pcol, r, f) for r, f in piv.columns]
        pivot_frames.append(piv)

    X_df = pd.concat(pivot_frames, axis=1)
    X_df = X_df.reindex(meta["word_idx"]).dropna(axis=1, how="all")

    return X_df.values.astype(float), meta.reset_index(drop=True)


# ── LOTO Ridge: train on y_train, predict X_test, evaluate against y_test ────
def loto_ridge_cross(X, y_train_col, y_test_col, trials):
    """
    Leave-one-trial-out Ridge regression with optional cross-target evaluation.

    Parameters
    ----------
    X           : feature matrix (n_words, n_features)
    y_train_col : target used to TRAIN the model  (SC_t or TC_t)
    y_test_col  : target used to EVALUATE predictions (SC_t or TC_t)
    trials      : trial index per word

    Returns
    -------
    preds : predicted values aligned to words where y_test_col is valid,
            or None if underpowered
    y_eval: the corresponding ground-truth y_test_col values
    """
    unique_trials = np.unique(trials)
    if len(unique_trials) < 3:
        return None, None
    if np.nanstd(y_train_col) == 0:
        return None, None

    preds  = np.full(len(y_train_col), np.nan)
    y_eval = y_test_col.copy()

    for held_trial in unique_trials:
        test_mask  = trials == held_trial
        train_mask = ~test_mask

        X_train       = X[train_mask]
        y_train       = y_train_col[train_mask]
        X_test        = X[test_mask]

        # filter training rows: need finite y_train
        finite_mask = np.isfinite(y_train)
        if finite_mask.sum() < 3:
            continue
        X_train = X_train[finite_mask]
        y_train = y_train[finite_mask]

        if np.std(y_train) == 0:
            continue

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("ridge",   RidgeCV(alphas=ALPHAS, cv=None)),
        ])
        pipe.fit(X_train, y_train)
        preds[test_mask] = pipe.predict(X_test)

    return preds, y_eval


# ── Per-subject worker ────────────────────────────────────────────────────────
def run_subject(subj, subj_df):
    rows = []
    print("  Starting: {}".format(subj), flush=True)

    for feat_name, power_cols in FEATURE_SETS.items():
        X, meta = build_word_feature_matrix(subj_df, power_cols)
        trials  = meta["trial"].values

        for train_target, test_target in DECODE_PAIRS:
            decode_type = "within" if train_target == test_target else "cross"

            y_train_col = meta[train_target].values.astype(float)
            y_test_col  = meta[test_target].values.astype(float)

            # Need valid rows for both train target and test target
            valid = np.isfinite(y_train_col) & np.isfinite(y_test_col)
            if valid.sum() < 6:
                rows.append(_nan_row(subj, feat_name, train_target, test_target,
                                     decode_type, int(valid.sum())))
                continue

            preds, y_eval = loto_ridge_cross(
                X[valid],
                y_train_col[valid],
                y_test_col[valid],
                trials[valid]
            )

            if preds is None:
                rows.append(_nan_row(subj, feat_name, train_target, test_target,
                                     decode_type, int(valid.sum())))
                continue

            # Evaluate on words where both prediction and test-target are finite
            ok = np.isfinite(preds) & np.isfinite(y_eval)
            if (ok.sum() < 4
                    or np.std(preds[ok]) == 0
                    or np.std(y_eval[ok]) == 0):
                rows.append(_nan_row(subj, feat_name, train_target, test_target,
                                     decode_type, int(valid.sum())))
                continue

            r, p = pearsonr(y_eval[ok], preds[ok])
            r2   = r ** 2

            print("  {} | {} | {}→{} ({}) r={:+.3f}  R²={:.3f}  p={:.4f}".format(
                subj, feat_name, train_target, test_target, decode_type,
                r, r2, p), flush=True)

            rows.append({
                "subject":      subj,
                "feature_set":  feat_name,
                "train_target": train_target,
                "test_target":  test_target,
                "decode_type":  decode_type,
                "n_words":      int(ok.sum()),
                "r":            r,
                "r2":           r2,
                "p":            p,
            })

    return rows


def _nan_row(subj, feat_name, train_target, test_target, decode_type, n):
    return {
        "subject":      subj,
        "feature_set":  feat_name,
        "train_target": train_target,
        "test_target":  test_target,
        "decode_type":  decode_type,
        "n_words":      n,
        "r":            np.nan,
        "r2":           np.nan,
        "p":            np.nan,
    }


# ── Main loop (parallel over subjects) ───────────────────────────────────────
all_rows = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(run_subject)(subj, df[df["subject"] == subj].copy())
    for subj in subjects
)

results = [row for subj_rows in all_rows for row in subj_rows]
results_df = pd.DataFrame(results)
results_df.to_csv("decoding_cross_sc_tc.csv", index=False)

# ── Group summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("GROUP-LEVEL SUMMARY  (one-sample t vs r = 0)")
print("=" * 75)

header = "  {:<20} {:<12} {:>4} {:>8} {:>7} {:>7} {:>8} {:>7}".format(
    "Feature set", "Condition", "n", "Mean r", "SD", "t", "p", "n_sig")
divider = "  " + "-" * 73

for feat_name in FEATURE_SETS:
    print("\n  Feature set: {}".format(feat_name))
    print(header)
    print(divider)

    for train_target, test_target in DECODE_PAIRS:
        decode_type = "within" if train_target == test_target else "cross"
        label       = "{}→{}  [{}]".format(train_target, test_target, decode_type)

        sub = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == train_target) &
            (results_df["test_target"]  == test_target)
        ].dropna(subset=["r"])

        if len(sub) < 2:
            print("  {:<20} {:<12}  (insufficient data)".format(feat_name, label))
            continue

        rs    = sub["r"].values
        t, p  = ttest_1samp(rs, popmean=0)
        n_sig = int((sub["p"] < 0.05).sum())

        print("  {:<20} {:<20} {:>4} {:>8.3f} {:>7.3f} {:>7.3f} {:>8.4f} {:>4}/{}".format(
            feat_name, label, len(rs), rs.mean(), rs.std(), t, p,
            n_sig, len(rs)))

# ── Cross vs Within comparison (paired t) ────────────────────────────────────
print("\n" + "=" * 75)
print("CROSS vs WITHIN  (paired t-test per feature set × direction)")
print("=" * 75)

for feat_name in FEATURE_SETS:
    print("\n  Feature set: {}".format(feat_name))
    print("  {:<25} {:>7} {:>7} {:>8}".format("Comparison", "t", "p", "n_subj"))
    print("  " + "-" * 50)

    # SC→SC vs SC→TC
    for within_pair, cross_pair in [
        (("SC_t", "SC_t"), ("SC_t", "TC_t")),
        (("TC_t", "TC_t"), ("TC_t", "SC_t")),
    ]:
        within_df = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == within_pair[0]) &
            (results_df["test_target"]  == within_pair[1])
        ].dropna(subset=["r"]).set_index("subject")

        cross_df = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == cross_pair[0]) &
            (results_df["test_target"]  == cross_pair[1])
        ].dropna(subset=["r"]).set_index("subject")

        shared = within_df.index.intersection(cross_df.index)
        if len(shared) < 3:
            continue

        r_within = within_df.loc[shared, "r"].values
        r_cross  = cross_df.loc[shared, "r"].values
        t, p     = ttest_rel(r_within, r_cross)
        label    = "{}→{} vs {}→{}".format(
            within_pair[0], within_pair[1], cross_pair[0], cross_pair[1])

        print("  {:<25} {:>7.3f} {:>7.4f} {:>8}".format(label, t, p, len(shared)))

print()
print("Saved → decoding_cross_sc_tc.csv")

Loaded 27162 rows | 26 subjects
Regions : ['LHP', 'RHP']
Freqs   : [3.0, 3.74, 4.67, 5.82, 7.26, 9.05, 11.28, 14.07, 17.55, 21.88, 27.29, 34.03, 42.44, 52.92, 66.0, 82.31, 102.64, 128.0]



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    4.1s



GROUP-LEVEL SUMMARY  (one-sample t vs r = 0)

  Feature set: encoding_only
  Feature set          Condition       n   Mean r      SD       t        p   n_sig
  -------------------------------------------------------------------------
  encoding_only        SC_t→SC_t  [within]    23   -0.449   0.317  -6.636   0.0000   14/23
  encoding_only        TC_t→TC_t  [within]    23   -0.355   0.361  -4.613   0.0001   13/23
  encoding_only        SC_t→TC_t  [cross]     23   -0.021   0.338  -0.296   0.7697    6/23
  encoding_only        TC_t→SC_t  [cross]     23    0.007   0.271   0.117   0.9075    4/23

  Feature set: retrieval_only
  Feature set          Condition       n   Mean r      SD       t        p   n_sig
  -------------------------------------------------------------------------
  retrieval_only       SC_t→SC_t  [within]    23   -0.412   0.226  -8.554   0.0000   14/23
  retrieval_only       TC_t→TC_t  [within]    23   -0.452   0.300  -7.067   0.0000   15/23
  retrieval_only       SC_t→T

[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    5.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    5.0s finished


In [21]:
for subj in subjects:
    sdf = df[df["subject"]==subj]
    words = sdf.groupby(["trial","recalled_word"])["SC_t"].first().reset_index()
    
    # trial-level mean SC_t
    trial_means = words.groupby("trial")["SC_t"].mean()
    print(subj, 
          "SC_t std={:.3f}".format(words["SC_t"].std()),
          "trial_mean_std={:.3f}".format(trial_means.std()),
          "ratio={:.2f}".format(trial_means.std()/words["SC_t"].std()))

R1494D SC_t std=1.390 trial_mean_std=1.170 ratio=0.84
R1501J SC_t std=0.994 trial_mean_std=0.979 ratio=0.98
R1502D SC_t std=1.626 trial_mean_std=1.032 ratio=0.63
R1503E SC_t std=1.840 trial_mean_std=1.155 ratio=0.63
R1504E SC_t std=0.341 trial_mean_std=0.398 ratio=1.17
R1509E SC_t std=2.187 trial_mean_std=2.285 ratio=1.04
R1521E SC_t std=0.845 trial_mean_std=1.004 ratio=1.19
R1523E SC_t std=0.520 trial_mean_std=0.674 ratio=1.30
R1532T SC_t std=0.832 trial_mean_std=0.964 ratio=1.16
R1536J SC_t std=2.925 trial_mean_std=1.200 ratio=0.41
R1537T SC_t std=0.971 trial_mean_std=0.567 ratio=0.58
R1538E SC_t std=nan trial_mean_std=nan ratio=nan
R1542J SC_t std=1.600 trial_mean_std=1.392 ratio=0.87
R1543E SC_t std=2.261 trial_mean_std=0.955 ratio=0.42
R1546E SC_t std=1.470 trial_mean_std=1.249 ratio=0.85
R1551T SC_t std=18.355 trial_mean_std=7.313 ratio=0.40
R1554T SC_t std=1.426 trial_mean_std=0.898 ratio=0.63
R1560T SC_t std=0.625 trial_mean_std=0.685 ratio=1.10
R1561E SC_t std=1.796 trial_mean

In [22]:
"""
Within-subject decoding + cross-decoding: SC_t ↔ TC_t from word-level iEEG IRASA features.

CV method  : Leave-One-Trial-Out per subject
Model      : RidgeCV (alpha selected by inner LOO-CV on training fold)
Features   : word-level encoding/retrieval frac_ssl and osc_ssl
             pivoted across regions x frequencies
Targets    : SC_t, TC_t (continuous, per recalled word)
             → within-trial demeaned before decoding (removes trial-mean leakage)

Decoding modes
──────────────
  within  : train on SC_t → test on SC_t  (and TC_t → TC_t)
  cross   : train on SC_t → test on TC_t  (and TC_t → SC_t)

Statistics : Pearson r, R², p-value per subject → group one-sample t vs r = 0
             Cross vs within comparison: paired t-test per feature set

Diagnostics (saved to decoding_diagnostics/)
────────────────────────────────────────────
  1. per_subject_r_stripplot.png   — per-subject r for all conditions × feature sets
  2. within_vs_cross_scatter.png   — within r vs cross r per subject (SC and TC)
  3. target_variance_plot.png      — SC_t / TC_t std and trial-mean ratio per subject
  4. prediction_scatter_grid.png   — predicted vs actual scatter for each subject
                                     (combined feature set, SC_t→SC_t only)
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import os

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from scipy.stats import pearsonr, ttest_1samp, ttest_rel
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH   = "ALL_SUBJECTS_irasa_labeled.csv"
RANDOM_SEED = 42
N_JOBS      = -1
DIAG_DIR    = "decoding_diagnostics"
os.makedirs(DIAG_DIR, exist_ok=True)

FEATURE_SETS = {
    "encoding_only":  ["encoding_frac_ssl",  "encoding_osc_ssl"],
    "retrieval_only": ["retrieval_frac_ssl",  "retrieval_osc_ssl"],
    "combined":       ["encoding_frac_ssl",   "encoding_osc_ssl",
                       "retrieval_frac_ssl",  "retrieval_osc_ssl"],
}

DECODE_PAIRS = [
    ("SC_t", "SC_t"),
    ("TC_t", "TC_t"),
    ("SC_t", "TC_t"),
    ("TC_t", "SC_t"),
]

ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df["region"] = df["region"].astype(str)

# ── Within-trial demeaning ────────────────────────────────────────────────────
# Removes trial-level mean from SC_t and TC_t within each subject × trial.
# This prevents the Ridge from learning trial-mean differences rather than
# word-level neural signals, which was the likely cause of negative r.
print("Applying within-trial demeaning to SC_t and TC_t ...")
for target in ["SC_t", "TC_t"]:
    df[target] = df.groupby(["subject", "trial"])[target].transform(
        lambda x: x - x.mean()
    )
print("Done.\n")

subjects = sorted(df["subject"].unique())
print("Loaded {} rows | {} subjects".format(len(df), len(subjects)))
print("Regions : {}".format(sorted(df["region"].unique())))
print("Freqs   : {}".format(sorted(df["freq_hz"].unique())))
print()


# ── Feature builder ───────────────────────────────────────────────────────────
def build_word_feature_matrix(subj_df, power_cols):
    meta = (
        subj_df
        .groupby(["trial", "recalled_word"])[["SC_t", "TC_t"]]
        .first()
        .reset_index()
        .reset_index(drop=True)
    )
    meta["word_idx"] = meta.index

    subj_df = subj_df.merge(
        meta[["trial", "recalled_word", "word_idx"]],
        on=["trial", "recalled_word"],
        how="left"
    )

    pivot_frames = []
    for pcol in power_cols:
        piv = subj_df.pivot_table(
            index="word_idx",
            columns=["region", "freq_hz"],
            values=pcol,
            aggfunc="mean"
        )
        piv.columns = ["{}_{}_{}Hz".format(pcol, r, f) for r, f in piv.columns]
        pivot_frames.append(piv)

    X_df = pd.concat(pivot_frames, axis=1)
    X_df = X_df.reindex(meta["word_idx"]).dropna(axis=1, how="all")

    return X_df.values.astype(float), meta.reset_index(drop=True)


# ── LOTO Ridge ────────────────────────────────────────────────────────────────
def loto_ridge_cross(X, y_train_col, y_test_col, trials):
    unique_trials = np.unique(trials)
    if len(unique_trials) < 3:
        return None, None
    if np.nanstd(y_train_col) == 0:
        return None, None

    preds  = np.full(len(y_train_col), np.nan)
    y_eval = y_test_col.copy()

    for held_trial in unique_trials:
        test_mask  = trials == held_trial
        train_mask = ~test_mask

        X_train = X[train_mask]
        y_train = y_train_col[train_mask]
        X_test  = X[test_mask]

        finite_mask = np.isfinite(y_train)
        if finite_mask.sum() < 3:
            continue
        X_train = X_train[finite_mask]
        y_train = y_train[finite_mask]

        if np.std(y_train) == 0:
            continue

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("ridge",   RidgeCV(alphas=ALPHAS, cv=None)),
        ])
        pipe.fit(X_train, y_train)
        preds[test_mask] = pipe.predict(X_test)

    return preds, y_eval


# ── Per-subject worker ────────────────────────────────────────────────────────
# Also collects (y_actual, y_pred) for diagnostic scatter plots
_scatter_store = {}   # keyed by subject; filled during run

def run_subject(subj, subj_df):
    rows         = []
    scatter_data = {}   # (feat_name, train_t, test_t) → (y_actual, y_pred)
    print("  Starting: {}".format(subj), flush=True)

    for feat_name, power_cols in FEATURE_SETS.items():
        X, meta = build_word_feature_matrix(subj_df, power_cols)
        trials  = meta["trial"].values

        for train_target, test_target in DECODE_PAIRS:
            decode_type = "within" if train_target == test_target else "cross"

            y_train_col = meta[train_target].values.astype(float)
            y_test_col  = meta[test_target].values.astype(float)

            valid = np.isfinite(y_train_col) & np.isfinite(y_test_col)
            if valid.sum() < 6:
                rows.append(_nan_row(subj, feat_name, train_target, test_target,
                                     decode_type, int(valid.sum())))
                continue

            preds, y_eval = loto_ridge_cross(
                X[valid], y_train_col[valid], y_test_col[valid], trials[valid]
            )

            if preds is None:
                rows.append(_nan_row(subj, feat_name, train_target, test_target,
                                     decode_type, int(valid.sum())))
                continue

            ok = np.isfinite(preds) & np.isfinite(y_eval)
            if (ok.sum() < 4
                    or np.std(preds[ok]) == 0
                    or np.std(y_eval[ok]) == 0):
                rows.append(_nan_row(subj, feat_name, train_target, test_target,
                                     decode_type, int(valid.sum())))
                continue

            r, p = pearsonr(y_eval[ok], preds[ok])
            r2   = r ** 2

            # store for scatter diagnostics
            scatter_data[(feat_name, train_target, test_target)] = (
                y_eval[ok], preds[ok]
            )

            print("  {} | {} | {}→{} ({}) r={:+.3f}  R²={:.3f}  p={:.4f}".format(
                subj, feat_name, train_target, test_target, decode_type,
                r, r2, p), flush=True)

            rows.append({
                "subject":      subj,
                "feature_set":  feat_name,
                "train_target": train_target,
                "test_target":  test_target,
                "decode_type":  decode_type,
                "n_words":      int(ok.sum()),
                "r":            r,
                "r2":           r2,
                "p":            p,
            })

    return rows, scatter_data


def _nan_row(subj, feat_name, train_target, test_target, decode_type, n):
    return {
        "subject":      subj,
        "feature_set":  feat_name,
        "train_target": train_target,
        "test_target":  test_target,
        "decode_type":  decode_type,
        "n_words":      n,
        "r":            np.nan,
        "r2":           np.nan,
        "p":            np.nan,
    }


# ── Main loop ─────────────────────────────────────────────────────────────────
outputs = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(run_subject)(subj, df[df["subject"] == subj].copy())
    for subj in subjects
)

all_rows     = [row  for rows, _  in outputs for row  in rows]
scatter_store = {subj: sdata for subj, (_, sdata) in zip(subjects, outputs)}

results_df = pd.DataFrame(all_rows)
results_df.to_csv("decoding_cross_sc_tc_demeaned.csv", index=False)


# ── Group summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("GROUP-LEVEL SUMMARY  (one-sample t vs r = 0)  [within-trial demeaned]")
print("=" * 75)

header  = "  {:<20} {:<22} {:>4} {:>8} {:>7} {:>7} {:>8} {:>7}".format(
    "Feature set", "Condition", "n", "Mean r", "SD", "t", "p", "n_sig")
divider = "  " + "-" * 75

for feat_name in FEATURE_SETS:
    print("\n  Feature set: {}".format(feat_name))
    print(header)
    print(divider)
    for train_target, test_target in DECODE_PAIRS:
        decode_type = "within" if train_target == test_target else "cross"
        label = "{}→{}  [{}]".format(train_target, test_target, decode_type)
        sub = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == train_target) &
            (results_df["test_target"]  == test_target)
        ].dropna(subset=["r"])
        if len(sub) < 2:
            continue
        rs   = sub["r"].values
        t, p = ttest_1samp(rs, popmean=0)
        n_sig = int((sub["p"] < 0.05).sum())
        print("  {:<20} {:<22} {:>4} {:>8.3f} {:>7.3f} {:>7.3f} {:>8.4f} {:>4}/{}".format(
            feat_name, label, len(rs), rs.mean(), rs.std(), t, p,
            n_sig, len(rs)))

print("\n" + "=" * 75)
print("CROSS vs WITHIN  (paired t-test)")
print("=" * 75)
for feat_name in FEATURE_SETS:
    print("\n  Feature set: {}".format(feat_name))
    print("  {:<28} {:>7} {:>7} {:>8}".format("Comparison", "t", "p", "n_subj"))
    print("  " + "-" * 53)
    for within_pair, cross_pair in [
        (("SC_t", "SC_t"), ("SC_t", "TC_t")),
        (("TC_t", "TC_t"), ("TC_t", "SC_t")),
    ]:
        w_df = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == within_pair[0]) &
            (results_df["test_target"]  == within_pair[1])
        ].dropna(subset=["r"]).set_index("subject")
        c_df = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == cross_pair[0]) &
            (results_df["test_target"]  == cross_pair[1])
        ].dropna(subset=["r"]).set_index("subject")
        shared = w_df.index.intersection(c_df.index)
        if len(shared) < 3:
            continue
        t, p = ttest_rel(w_df.loc[shared, "r"].values,
                         c_df.loc[shared, "r"].values)
        label = "{}→{} vs {}→{}".format(*within_pair, *cross_pair)
        print("  {:<28} {:>7.3f} {:>7.4f} {:>8}".format(label, t, p, len(shared)))

print()
print("Saved → decoding_cross_sc_tc_demeaned.csv")


# ════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC PLOTS
# ════════════════════════════════════════════════════════════════════════════

PALETTE = {
    "SC_t→SC_t": "#2166ac",
    "TC_t→TC_t": "#d6604d",
    "SC_t→TC_t": "#74add1",
    "TC_t→SC_t": "#f4a582",
}
FS_LABELS = list(FEATURE_SETS.keys())


# ── Plot 1: Per-subject r stripplot ──────────────────────────────────────────
print("Saving diagnostic plot 1/4: per_subject_r_stripplot.png ...")

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
fig.suptitle("Per-subject decoding r  [within-trial demeaned]",
             fontsize=13, fontweight="bold", y=1.01)

rng = np.random.default_rng(RANDOM_SEED)

for ax, feat_name in zip(axes, FS_LABELS):
    for xi, (train_t, test_t) in enumerate(DECODE_PAIRS):
        key   = "{}→{}".format(train_t, test_t)
        color = PALETTE[key]
        sub   = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == train_t) &
            (results_df["test_target"]  == test_t)
        ].dropna(subset=["r"])
        rs = sub["r"].values
        jitter = rng.uniform(-0.15, 0.15, size=len(rs))
        ax.scatter(np.full(len(rs), xi) + jitter, rs,
                   color=color, alpha=0.7, s=40, zorder=3)
        # mean bar
        ax.hlines(rs.mean(), xi - 0.3, xi + 0.3,
                  colors=color, linewidths=2.5, zorder=4)

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(range(len(DECODE_PAIRS)))
    ax.set_xticklabels(
        ["{}→{}".format(a, b) for a, b in DECODE_PAIRS],
        rotation=35, ha="right", fontsize=8
    )
    ax.set_title(feat_name, fontsize=10)
    ax.set_xlabel("Decode condition")
    if ax == axes[0]:
        ax.set_ylabel("Pearson r")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DIAG_DIR, "per_subject_r_stripplot.png"),
            dpi=150, bbox_inches="tight")
plt.close()


# ── Plot 2: Within vs Cross scatter ──────────────────────────────────────────
print("Saving diagnostic plot 2/4: within_vs_cross_scatter.png ...")

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle("Within r vs Cross r per subject  [within-trial demeaned]",
             fontsize=13, fontweight="bold")

for col, feat_name in enumerate(FS_LABELS):
    for row, (within_pair, cross_pair) in enumerate([
        (("SC_t", "SC_t"), ("SC_t", "TC_t")),
        (("TC_t", "TC_t"), ("TC_t", "SC_t")),
    ]):
        ax = axes[row, col]
        w_df = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == within_pair[0]) &
            (results_df["test_target"]  == within_pair[1])
        ].dropna(subset=["r"]).set_index("subject")
        c_df = results_df[
            (results_df["feature_set"]  == feat_name) &
            (results_df["train_target"] == cross_pair[0]) &
            (results_df["test_target"]  == cross_pair[1])
        ].dropna(subset=["r"]).set_index("subject")
        shared = w_df.index.intersection(c_df.index)

        r_w = w_df.loc[shared, "r"].values
        r_c = c_df.loc[shared, "r"].values

        within_key = "{}→{}".format(*within_pair)
        cross_key  = "{}→{}".format(*cross_pair)
        ax.scatter(r_w, r_c,
                   color=PALETTE[within_key], edgecolors="black",
                   linewidths=0.5, s=55, alpha=0.85)

        # annotate subject labels
        for subj, xv, yv in zip(shared, r_w, r_c):
            ax.annotate(subj, (xv, yv), fontsize=5,
                        xytext=(3, 3), textcoords="offset points", alpha=0.7)

        lim = max(abs(np.concatenate([r_w, r_c])).max() * 1.15, 0.2)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        ax.axvline(0, color="gray", lw=0.8, ls="--")
        ax.plot([-lim, lim], [-lim, lim], color="gray", lw=0.6, ls=":")
        ax.set_xlabel("r  ({})".format(within_key), fontsize=8)
        ax.set_ylabel("r  ({})".format(cross_key),  fontsize=8)
        if row == 0:
            ax.set_title(feat_name, fontsize=10)
        ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(DIAG_DIR, "within_vs_cross_scatter.png"),
            dpi=150, bbox_inches="tight")
plt.close()


# ── Plot 3: Target variance diagnostics ──────────────────────────────────────
print("Saving diagnostic plot 3/4: target_variance_plot.png ...")

diag_rows = []
for subj in subjects:
    sdf   = df[df["subject"] == subj]
    words = (
        sdf.groupby(["trial", "recalled_word"])[["SC_t", "TC_t"]]
        .first()
        .reset_index()
    )
    for tgt in ["SC_t", "TC_t"]:
        total_std    = words[tgt].std()
        trial_means  = words.groupby("trial")[tgt].mean()
        between_std  = trial_means.std()
        ratio        = between_std / total_std if total_std > 0 else np.nan
        diag_rows.append({
            "subject":      subj,
            "target":       tgt,
            "total_std":    total_std,
            "between_std":  between_std,
            "ratio":        ratio,
        })

diag_df = pd.DataFrame(diag_rows)
subj_order = sorted(subjects)
x = np.arange(len(subj_order))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle("Target variance diagnostics  [after within-trial demeaning]",
             fontsize=12, fontweight="bold")

for ax, tgt, color in zip(axes, ["SC_t", "TC_t"], ["#2166ac", "#d6604d"]):
    sub = diag_df[diag_df["target"] == tgt].set_index("subject").reindex(subj_order)
    ax.bar(x, sub["total_std"].values,   color=color,   alpha=0.6, label="Total std")
    ax.bar(x, sub["between_std"].values, color=color,   alpha=1.0,
           label="Between-trial std", width=0.4)
    ax2 = ax.twinx()
    ax2.plot(x, sub["ratio"].values, "o--", color="black",
             markersize=5, linewidth=1, label="Ratio (between/total)")
    ax2.axhline(1.0, color="black", lw=0.7, ls=":")
    ax2.set_ylabel("Ratio", fontsize=8)
    ax2.set_ylim(0, 2)
    ax.set_ylabel("Std  ({})".format(tgt), fontsize=9)
    ax.set_title(tgt, fontsize=10)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="upper right")
    ax.grid(axis="y", alpha=0.3)

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(subj_order, rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(DIAG_DIR, "target_variance_plot.png"),
            dpi=150, bbox_inches="tight")
plt.close()


# ── Plot 4: Predicted vs actual scatter grid (combined, SC_t→SC_t) ───────────
print("Saving diagnostic plot 4/4: prediction_scatter_grid.png ...")

valid_subjs = [
    s for s in subjects
    if ("combined", "SC_t", "SC_t") in scatter_store.get(s, {})
]

ncols = 5
nrows = int(np.ceil(len(valid_subjs) / ncols))
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 3, nrows * 3))
axes = np.array(axes).flatten()
fig.suptitle("Predicted vs Actual SC_t  [combined features, within-trial demeaned]",
             fontsize=12, fontweight="bold")

for i, subj in enumerate(valid_subjs):
    ax = axes[i]
    y_actual, y_pred = scatter_store[subj][("combined", "SC_t", "SC_t")]
    r, p = pearsonr(y_actual, y_pred)
    ax.scatter(y_actual, y_pred, s=18, alpha=0.5,
               color="#2166ac", edgecolors="none")
    # regression line
    m, b = np.polyfit(y_actual, y_pred, 1)
    xs = np.linspace(y_actual.min(), y_actual.max(), 50)
    ax.plot(xs, m * xs + b, color="#d6604d", lw=1.5)
    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.axvline(0, color="gray", lw=0.5, ls="--")
    ax.set_title("{}\nr={:+.2f} p={:.3f}".format(subj, r, p), fontsize=7)
    ax.set_xlabel("Actual", fontsize=7)
    ax.set_ylabel("Predicted", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(alpha=0.2)

# hide unused panels
for j in range(len(valid_subjs), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(DIAG_DIR, "prediction_scatter_grid.png"),
            dpi=150, bbox_inches="tight")
plt.close()

print("\nAll diagnostic plots saved to: {}/".format(DIAG_DIR))
print("  1. per_subject_r_stripplot.png")
print("  2. within_vs_cross_scatter.png")
print("  3. target_variance_plot.png")
print("  4. prediction_scatter_grid.png")

Applying within-trial demeaning to SC_t and TC_t ...
Done.

Loaded 27162 rows | 26 subjects
Regions : ['LHP', 'RHP']
Freqs   : [3.0, 3.74, 4.67, 5.82, 7.26, 9.05, 11.28, 14.07, 17.55, 21.88, 27.29, 34.03, 42.44, 52.92, 66.0, 82.31, 102.64, 128.0]



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:    2.4s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    5.3s
[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    6.3s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  26 out of  26 | elapsed:    6.3s finished



GROUP-LEVEL SUMMARY  (one-sample t vs r = 0)  [within-trial demeaned]

  Feature set: encoding_only
  Feature set          Condition                 n   Mean r      SD       t        p   n_sig
  ---------------------------------------------------------------------------
  encoding_only        SC_t→SC_t  [within]      23   -0.249   0.370  -3.155   0.0046    7/23
  encoding_only        TC_t→TC_t  [within]      22   -0.116   0.459  -1.164   0.2576    9/22
  encoding_only        SC_t→TC_t  [cross]       22    0.087   0.209   1.917   0.0689    4/22
  encoding_only        TC_t→SC_t  [cross]       22    0.115   0.235   2.252   0.0351    7/22

  Feature set: retrieval_only
  Feature set          Condition                 n   Mean r      SD       t        p   n_sig
  ---------------------------------------------------------------------------
  retrieval_only       SC_t→SC_t  [within]      23   -0.204   0.277  -3.451   0.0023    6/23
  retrieval_only       TC_t→TC_t  [within]      22   -0.078  